# 12 — Per-acquisition Sentinel-1 and daily rainfall, 2022–2025

**Status: SOURCE WORKFLOW ONLY — not executed or validated.** Running this notebook requires Earth Engine access and a local project boundary. It exports actual results only after successful extraction; there are no invented per-scene results in the committed file.

Study question: can individual Sentinel-1 scenes distinguish the 2022, 2024 and 2025 high-June screening extents (with 2023 as a low-June comparator) when monthly medians cannot?

- Compare Sentinel-1 ascending orbit 41 and descending orbit 77 **separately**; never combine VV intensities as if geometry is equivalent.
- Per-scene fixed −19/−17/−15 dB VV thresholds, incidence-angle screen, 30 m focal median with ORIGINAL image support mask preserved.
- Report acquisition timestamps in UTC and Asia/Dhaka. CHIRPS daily image DATE LABELS are not assumed to equal Bangladesh midnight-to-midnight rainfall totals. Rainfall-date offsets of −1/0/+1 day are sensitivity alternatives, not corrections.
- Area is LOW-VV SCREENING, not validated water extent, depth, damage or flood severity. First/last observed detections do not resolve onset or recession between passes.
- Inputs: study-area boundary, original Notebook 09 scene inventory, Earth Engine Sentinel-1 GRD and CHIRPS daily. Outputs: CSVs, figures, QA report and optional separate per-orbit frequency maps.


## 12A — Configuration, boundary, Earth Engine and reproducibility

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import ee

start = Path(os.environ.get("HAKALUKI_ROOT", Path.cwd())).expanduser().resolve()
for candidate in [start, *start.parents]:
    if ((candidate/"data/boundary/hakaluki_boundary.gpkg").is_file()
        or (candidate/"Study area/Hakaluki_Haor.shp").is_file()):
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Set HAKALUKI_ROOT to the research project or start Jupyter inside it")
sys.path.insert(0, str(ROOT))
from src.scene12_utils import normalize_daily_rain, attach_rain_to_scenes, check_scene_areas, describe_scene_support

BOUNDARY = ROOT/"data/boundary/hakaluki_boundary.gpkg"
if not BOUNDARY.is_file():
    BOUNDARY = ROOT/"Study area/Hakaluki_Haor.shp"
TABLES, FIGURES, CACHE = ROOT/"results/tables", ROOT/"results/figures", ROOT/"data/processed/scene12_cache"
for folder in (TABLES, FIGURES, CACHE):
    folder.mkdir(parents=True, exist_ok=True)

YEARS = (2022, 2023, 2024, 2025)
MONTHS = (5, 9)             # May through September inclusive
TRACKS = (("ASCENDING", 41), ("DESCENDING", 77))
THRESHOLDS_DB = (-19.0, -17.0, -15.0)
ANGLE_RANGE = (30.0, 45.0)
FILTER_RADIUS_M, SCALE_M = 30, 30
MIN_SCENE_COVERAGE = 0.90
FORCE_RECOMPUTE = False
DO_VV_VH_SAMPLES = True

haor = gpd.read_file(BOUNDARY).to_crs("EPSG:4326")
assert len(haor) and haor.geometry.notna().all() and haor.geometry.is_valid.all()
polygon = haor.geometry.union_all()
project = os.getenv("EE_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
try:
    if project:
        ee.Initialize(project=project)
    else:
        ee.Initialize()
except Exception as err:
    raise RuntimeError("Initialize Earth Engine and set EE_PROJECT locally if required; do not publish credentials") from err
ROI = ee.Geometry(polygon.__geo_interface__)
roi_stats = ee.Image.pixelArea().rename("roi_m2").reduceRegion(
    reducer=ee.Reducer.sum(), geometry=ROI, crs="EPSG:32646",
    scale=SCALE_M, maxPixels=100_000_000, tileScale=4).getInfo()
REFERENCE_KM2 = float(roi_stats["roi_m2"])/1e6
assert 10 < REFERENCE_KM2 < 500

CONFIG = {"years": list(YEARS), "months": list(MONTHS), "tracks": [list(x) for x in TRACKS],
          "thresholds": list(THRESHOLDS_DB), "angle": list(ANGLE_RANGE),
          "radius_m": FILTER_RADIUS_M, "scale_m": SCALE_M,
          "roi_wkb_hash": hashlib.sha256(polygon.wkb).hexdigest(),
          "source": "COPERNICUS/S1_GRD", "version": "12.1"}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
print("Project folder:", ROOT, "| EE reference area:", round(REFERENCE_KM2,3), "km²")
print("Configuration hash:", CONFIG_HASH, " | no new result until executed")


## 12B — Cross-check exact scene identities and acquisition times

The local Notebook 09 inventory must agree with the current Earth Engine catalog for both target orbits and every year. Mismatches abort the extraction rather than silently changing the sample. Local civil time conversion uses the named Asia/Dhaka timezone.

In [ ]:
candidate_inventories = [
    ROOT/"results/tables/inundation09_s1_scene_inventory.csv",
    ROOT/"docs/assets/tables/analysis_2019_2025/inundation09_s1_scene_inventory.csv"
]
inventory_path = next((p for p in candidate_inventories if p.exists()), None)
if inventory_path is None:
    raise FileNotFoundError("Run Notebook 09 first or copy its scene inventory into docs/assets/tables")
inventory = pd.read_csv(inventory_path)
necessary = {"scene_id","acquisition_utc","pass","relative_orbit"}
assert necessary.issubset(inventory.columns), f"Missing inventory columns: {necessary-set(inventory.columns)}"
inventory["acquisition_utc"] = pd.to_datetime(inventory.acquisition_utc,utc=True,errors="raise")
inventory["relative_orbit"] = pd.to_numeric(inventory.relative_orbit,errors="raise").astype(int)
inventory["year"] = inventory.acquisition_utc.dt.year
inventory["month"] = inventory.acquisition_utc.dt.month
inventory = inventory.loc[inventory.year.isin(YEARS) &
                          inventory.month.between(*MONTHS)].copy()
inventory = inventory.loc[[tuple(x) in TRACKS for x in
                           zip(inventory["pass"],inventory.relative_orbit)]]
if inventory.scene_id.duplicated().any():
    raise RuntimeError("Duplicate source scene IDs")
inventory["acquisition_local"] = inventory.acquisition_utc.dt.tz_convert("Asia/Dhaka")
inventory = inventory.sort_values("acquisition_utc").reset_index(drop=True)

s1 = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(ROI)
      .filter(ee.Filter.eq("instrumentMode","IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VH")))
def selected_collection(year, p, orbit):
    return (s1.filterDate(f"{year}-{MONTHS[0]:02d}-01",f"{year}-{MONTHS[1]+1:02d}-01")
            .filter(ee.Filter.eq("orbitProperties_pass",p))
            .filter(ee.Filter.eq("relativeOrbitNumber_start",orbit)))
checks = []
for year in YEARS:
    for p,orbit in TRACKS:
        catalog = selected_collection(year,p,orbit)
        n = int(catalog.size().getInfo())
        if n > 100:
            raise RuntimeError("Unexpected annual scene count; check Earth Engine filters")
        remote = set(catalog.aggregate_array("system:index").getInfo())
        local = set(inventory.loc[(inventory.year==year)&(inventory["pass"]==p)&
                                  (inventory.relative_orbit==orbit),"scene_id"])
        if len(remote)!=n or remote!=local:
            raise RuntimeError(f"EE/inventory mismatch for {year} {p}/{orbit}. "
                               f"Remote-only: {list(remote-local)[:3]}, local-only: {list(local-remote)[:3]}")
        checks.append({"year":year,"orbit_pass":p,"relative_orbit":orbit,"scene_count":n})
checks = pd.DataFrame(checks)
checks.to_csv(TABLES/"scene12_catalog_inventory_check.csv",index=False)
print("Catalog and Notebook 09 scene IDs match for the analysis window.")
display(checks)
display(inventory.loc[(inventory.year==2022)&inventory.month.isin([6,7]),
                  ["scene_id","acquisition_utc","acquisition_local","pass","relative_orbit"]])


## 12C — Compute per-scene VV screening extents, no monthly median

Use the original incidence-angle-masked VV support after focal filtering. All area numerators and support denominators use the same projected 30 m pixel-area reduction. Processing proceeds in small batches and maintains a configuration-specific local cache. VH remains available for later distribution QA; no uncalibrated VH classification is asserted.

In [ ]:
def prepared_polarizations(image):
    angle = image.select("angle")
    angle_ok = angle.gte(ANGLE_RANGE[0]).And(angle.lte(ANGLE_RANGE[1]))
    vv_original = image.select("VV").updateMask(angle_ok)
    vh_original = image.select("VH").updateMask(angle_ok)
    vv = vv_original.focal_median(radius=FILTER_RADIUS_M,kernelType="circle",units="meters")
    vh = vh_original.focal_median(radius=FILTER_RADIUS_M,kernelType="circle",units="meters")
    return (vv.updateMask(vv_original.mask()).rename("VV"),
            vh.updateMask(vh_original.mask()).rename("VH"))

def summarize_scene(scene_id):
    image = ee.Image("COPERNICUS/S1_GRD/"+scene_id)
    vv, vh = prepared_polarizations(image)
    observed = vv.mask().unmask(0).gt(0)
    pixel = ee.Image.pixelArea()
    bands = [pixel.multiply(observed).rename("observed_m2")]
    for cutoff,label in zip(THRESHOLDS_DB,["m19","m17","m15"]):
        flagged = vv.lte(cutoff).unmask(0).And(observed)
        bands.append(pixel.multiply(flagged).rename("area_"+label+"_m2"))
    areas = ee.Image.cat(bands).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=ROI, crs="EPSG:32646",
        scale=SCALE_M, maxPixels=100_000_000, tileScale=4)
    return ee.Feature(None,areas).set("scene_id",scene_id)

chunks = []
for year in YEARS:
    for p,orbit in TRACKS:
        source = inventory.loc[(inventory.year==year)&
                               (inventory["pass"]==p)&
                               (inventory.relative_orbit==orbit)]
        ids = source.scene_id.tolist()
        cache = CACHE / f"S1_{year}_{p}_{orbit}_{CONFIG_HASH}.csv"
        if cache.is_file() and not FORCE_RECOMPUTE:
            output = pd.read_csv(cache)
            if set(output.scene_id)!=set(ids):
                raise RuntimeError("Cache scene IDs differ from the current inventory")
        else:
            print(f"Extracting {year} {p} / orbit {orbit}: {len(ids)} scenes")
            rows = []
            for k in range(0,len(ids),8):
                group = [summarize_scene(sid) for sid in ids[k:k+8]]
                if group:
                    rows.extend(x["properties"] for x in ee.FeatureCollection(group).getInfo()["features"])
            output = pd.DataFrame(rows) if rows else pd.DataFrame(
                columns=["scene_id","observed_m2","area_m19_m2","area_m17_m2","area_m15_m2"])
            if set(output.scene_id)!=set(ids):
                raise RuntimeError("Server did not return all expected scenes")
            output.to_csv(cache,index=False)
        chunks.append(output)
area_data = pd.concat(chunks,ignore_index=True)
if area_data.scene_id.duplicated().any():
    raise RuntimeError("Duplicated Earth Engine outputs")
scenes = inventory.merge(area_data,on="scene_id",how="left",validate="one_to_one")
for field in ["observed_m2","area_m19_m2","area_m17_m2","area_m15_m2"]:
    scenes[field] = pd.to_numeric(scenes[field],errors="raise")
scenes["observed_km2"] = scenes.observed_m2 / 1e6
for key in ("m19","m17","m15"):
    scenes["lowvv_"+key+"_km2"] = scenes["area_"+key+"_m2"]/1e6
scenes["observed_fraction"] = scenes.observed_km2 / REFERENCE_KM2
scenes["eligible"] = scenes.observed_fraction.ge(MIN_SCENE_COVERAGE)
scenes["screened_fraction_of_observed"] = (
    scenes.lowvv_m17_km2 / scenes.observed_km2.replace(0,np.nan))
scenes["config_hash"] = CONFIG_HASH
check_scene_areas(scenes, REFERENCE_KM2)
if not scenes.observed_fraction.between(0,1.001).all():
    raise RuntimeError("Scene support fraction is outside accepted range")
scene_csv = TABLES/"scene12_S1_per_acquisition_2022_2025.csv"
scenes.to_csv(scene_csv,index=False)
scene_summary = describe_scene_support(scenes)
scene_summary.to_csv(TABLES/"scene12_pass_support_summary.csv",index=False)
print("Extracted scenes:",len(scenes),"| meeting 90% support:",int(scenes.eligible.sum()))
display(scene_summary)


## 12D — CHIRPS daily date labels and acquisition alignment

Rainfall is extracted from the same Earth Engine daily product in every compared year. The exact CHIRPS accumulation-day convention is not yet verified, so ±1-day label shifts are reported as sensitivity checks rather than treated as physical time corrections.

In [ ]:
def chirps_daily_year(year):
    series=(ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
            .filterDate(f"{year}-04-20",f"{year}-10-05")
            .sort("system:time_start"))
    n=int(series.size().getInfo())
    if not 150 <= n <= 190:
        raise RuntimeError(f"Unexpected daily rainfall image count: {year}: {n}")
    source=series.toList(n)
    results=[]
    for start in range(0,n,20):
        batch=[]
        for i in range(start,min(start+20,n)):
            image=ee.Image(source.get(i))
            value=image.select("precipitation").reduceRegion(
                reducer=ee.Reducer.mean(),geometry=ROI,
                scale=5566,maxPixels=1_000_000,tileScale=4)
            batch.append(ee.Feature(None,value).set("ms",image.get("system:time_start")))
        results += [f["properties"] for f in ee.FeatureCollection(batch).getInfo()["features"]]
    frame=pd.DataFrame(results)
    if len(frame)!=n or frame.precipitation.isna().any():
        raise RuntimeError("Missing CHIRPS daily values; no imputation")
    frame["date"]=pd.to_datetime(frame.ms,unit="ms",utc=True).dt.tz_localize(None).dt.normalize()
    frame["rain_mm_day"]=pd.to_numeric(frame.precipitation,errors="raise")
    frame["year"]=year
    return frame[["date","rain_mm_day","year"]].sort_values("date")

blocks=[]
for year in YEARS:
    checkpoint=CACHE/f"CHIRPS_{year}_{CONFIG_HASH}.csv"
    if checkpoint.exists() and not FORCE_RECOMPUTE:
        one=pd.read_csv(checkpoint,parse_dates=["date"])
    else:
        print("Daily CHIRPS extraction:",year)
        one=chirps_daily_year(year)
        one.to_csv(checkpoint,index=False)
    blocks.append(one)
rain_source=pd.concat(blocks,ignore_index=True)
if rain_source.date.duplicated().any():
    raise RuntimeError("Duplicate CHIRPS date labels")
rain=normalize_daily_rain(rain_source)
rain.to_csv(TABLES/"scene12_CHIRPS_daily_and_prior_windows.csv",index=False)
scenes=attach_rain_to_scenes(scenes,rain)
scenes.to_csv(scene_csv,index=False)
print("CHIRPS: date labels retained; physical accumulation windows not yet independently verified.")
display(scenes.loc[(scenes.year==2022)&(scenes.month==6),
    ["acquisition_utc","acquisition_local","pass","lowvv_m17_km2",
     "rain_previous_3d_mm_label_shift_-1","rain_previous_3d_mm_label_shift_+0",
     "rain_previous_3d_mm_label_shift_+1"]])


## 12E — Per-scene timeline and daily CHIRPS figure

The radar x coordinate is Asia/Dhaka local acquisition time; the CHIRPS x coordinate is the daily product date label. Both orbit passes remain separate. Comparison to rainfall is contextual, not causal.

In [ ]:
from matplotlib.dates import AutoDateLocator, DateFormatter
fig,axes=plt.subplots(2,2,figsize=(16,10),constrained_layout=True)
colors={"ASCENDING":"tab:blue","DESCENDING":"tab:orange"}
symbols={"ASCENDING":"o","DESCENDING":"s"}
for ax,year in zip(axes.flat,YEARS):
    first,last=pd.Timestamp(f"{year}-05-01"),pd.Timestamp(f"{year}-09-30")
    labelled=rain.loc[rain.date.between(first,last)]
    right=ax.twinx()
    right.bar(labelled.date,labelled.rain_mm_day,width=0.8,color="0.6",alpha=0.23)
    right.set_ylabel("Daily CHIRPS label (mm)",color="0.4")
    for p in ("ASCENDING","DESCENDING"):
        sample=scenes.loc[(scenes.year==year)&scenes.eligible&
                          (scenes["pass"]==p)].sort_values("acquisition_utc")
        times=sample.acquisition_local.dt.tz_localize(None)
        ax.scatter(times,sample.lowvv_m17_km2,color=colors[p],
                   marker=symbols[p],label=p,s=32,zorder=4)
        ax.plot(times,sample.lowvv_m17_km2,color=colors[p],alpha=0.4,lw=1)
    ax.set(title=str(year)+" — −17 dB per-scene VV screening",
           ylabel="Low-VV screening extent (km²)",
           xlabel="SAR acquisition in Bangladesh local time",
           ylim=(0,REFERENCE_KM2*1.04))
    ax.xaxis.set_major_locator(AutoDateLocator())
    ax.xaxis.set_major_formatter(DateFormatter("%d %b"))
    ax.tick_params(axis="x",rotation=30)
    ax.grid(alpha=0.15)
axes.flat[0].legend(loc="upper left")
fig.suptitle("Hakaluki Haor scene-scale radar and labelled CHIRPS rain: exploratory")
figure_file=FIGURES/"scene12_per_scene_2022_2025_CHIRPS.png"
fig.savefig(figure_file,dpi=170,bbox_inches="tight")
plt.show()
print("Saved",figure_file)

june=scenes.loc[(scenes.month==6)&scenes.eligible]
june_summary=(june.groupby(["year","pass"],as_index=False)
    .agg(n_scenes=("scene_id","nunique"),
         minimum_m17_km2=("lowvv_m17_km2","min"),
         maximum_m17_km2=("lowvv_m17_km2","max"),
         median_observation_fraction=("observed_fraction","median")))
june_summary.to_csv(TABLES/"scene12_june_scene_range_by_orbit.csv",index=False)
display(june_summary)


## 12F — Sampled VV and VH distribution checks

These images are numerical distribution examples, not reference labels or an optimized threshold. Inspect June 2022 and June 2024 from the same ascending orbit; repeat for descending and different locations if QA indicates geometry dependence.

In [ ]:
if DO_VV_VH_SAMPLES:
    selected_ids=[]
    for year in (2022,2024):
        candidates=inventory.loc[(inventory.year==year)&(inventory.month==6)&
                                 (inventory["pass"]=="ASCENDING")].sort_values("acquisition_utc")
        if len(candidates)>0:
            selected_ids.append(candidates.iloc[0].scene_id)
        if len(candidates)>1:
            selected_ids.append(candidates.iloc[-1].scene_id)
    rows=[]
    for scene_id in dict.fromkeys(selected_ids):
        image=ee.Image("COPERNICUS/S1_GRD/"+scene_id)
        vv,vh=prepared_polarizations(image)
        sample=ee.Image.cat([vv,vh]).sample(region=ROI,scale=60,
                                            numPixels=2500,seed=12,geometries=False)
        x=sample.aggregate_array("VV").getInfo()
        y=sample.aggregate_array("VH").getInfo()
        if len(x)!=len(y):
            raise RuntimeError("VV/VH sample count mismatch")
        year=int(inventory.loc[inventory.scene_id==scene_id,"year"].iloc[0])
        for u,v in zip(x,y):
            if u is not None and v is not None:
                rows.append({"scene_id":scene_id,"year":year,
                             "VV_db":float(u),"VH_db":float(v)})
    hist=pd.DataFrame(rows)
    if hist.empty:
        raise RuntimeError("No valid VV/VH sample points for QA")
    hist.to_csv(TABLES/"scene12_selected_scene_VV_VH_samples.csv",index=False)
    fig,axes=plt.subplots(1,2,figsize=(13,4.5),constrained_layout=True)
    for scene_id,subset in hist.groupby("scene_id"):
        short=scene_id[17:32]
        axes[0].hist(subset.VV_db,bins=np.arange(-35,6,1),density=True,
                     histtype="step",label=short)
        axes[1].hist(subset.VH_db,bins=np.arange(-45,1,1),density=True,
                     histtype="step",label=short)
    for ax,channel in zip(axes,["VV","VH"]):
        ax.set(xlabel=f"{channel} (dB)",ylabel="Sample density",
               title=f"{channel} pixel distributions — sample, not validation")
    for cutoff in THRESHOLDS_DB:
        axes[0].axvline(cutoff,color="0.5",alpha=0.35,linestyle="--")
    axes[0].legend(fontsize=8)
    fig.savefig(FIGURES/"scene12_VV_VH_sample_distributions.png",dpi=160)
    plt.show()
else:
    print("VV/VH sample distribution QA deferred by configuration")


## 12G — Optional per-pixel scene-normalized low-VV frequency maps

The denominator is the number of **actual valid scenes** at each pixel, separately for each pass/year. Missing scenes are never treated as dry. This is not a validated physical hydroperiod. Spatial exports are disabled by default; enable them after examining the scene-coverage QA.

In [ ]:
DOWNLOAD_SCENE_FREQUENCY_MAPS = False
if DOWNLOAD_SCENE_FREQUENCY_MAPS:
    import requests
    import rasterio
    out=ROOT/"results/rasters/scene12_lowvv_frequency"
    out.mkdir(parents=True,exist_ok=True)
    for year in (2022,2024,2025):
        for p,orbit in TRACKS:
            ids=inventory.loc[(inventory.year==year)&(inventory["pass"]==p)&
                               (inventory.relative_orbit==orbit),"scene_id"].tolist()
            if not ids:
                continue
            per_scene=[]
            for name in ids:
                vv,_=prepared_polarizations(ee.Image("COPERNICUS/S1_GRD/"+name))
                valid=vv.mask().unmask(0).gt(0)
                low=vv.lte(-17).unmask(0).And(valid)
                per_scene.append(ee.Image.cat([valid.toFloat().rename("n_valid"),
                                               low.toFloat().rename("n_low")]))
            sums=ee.ImageCollection.fromImages(per_scene).sum()
            observed=sums.select("n_valid")
            detected=sums.select("n_low")
            fraction=detected.divide(observed.max(1)).updateMask(observed.gt(0))
            raster=ee.Image.cat([fraction.rename("lowvv_frequency"),
                                 observed.rename("observed_scene_count")]).toFloat().clip(ROI)
            destination=out/f"scene12_{year}_{p}_{orbit}_{CONFIG_HASH}_30m.tif"
            if not destination.exists() or FORCE_RECOMPUTE:
                url=raster.getDownloadURL({
                    "region": ROI, "crs":"EPSG:32646","scale":SCALE_M,"format":"GEO_TIFF"})
                response=requests.get(url,timeout=180)
                response.raise_for_status()
                destination.write_bytes(response.content)
            with rasterio.open(destination) as src:
                if src.count!=2 or src.crs.to_epsg()!=32646:
                    raise RuntimeError("Downloaded scene frequency map failed QA")
            print("Verified",destination.name)
else:
    print("Per-pixel frequency GeoTIFF exports not yet enabled; scene-level CSV is available.")


## 12H — Export processing manifest, summary and limitations

Only executed observed data populate these tables. The report is a per-scene screening assessment, not a claim of detected flood depth, true hydroperiod or methane emission. Generated outputs remain local until reviewed and deliberately committed.

In [ ]:
from datetime import datetime, timezone
report_dir=ROOT/"results/reports"
report_dir.mkdir(parents=True,exist_ok=True)
manifest={
    "notebook":"12_scene_level_wetland_dynamics_2022_2025.ipynb",
    "generated_utc":datetime.now(timezone.utc).isoformat(),
    "configuration":CONFIG,"hash":CONFIG_HASH,
    "scene_count":int(len(scenes)),"eligible_scene_count":int(scenes.eligible.sum()),
    "reference_area_km2":REFERENCE_KM2,
    "source_scene_inventory":str(inventory_path.relative_to(ROOT)).replace("\\","/"),
    "status":"executed_exploratory_screening_not_validated_flood",
    "CHIRPS_daily_physical_accumulation_window_verified":False,
    "ascending_and_descending_radar_backscatter_averaged":False,
    "methane_emissions_estimated":False,
    "independent_water_validation_completed":False,
    "per_pixel_spatial_maps_generated":bool(DOWNLOAD_SCENE_FREQUENCY_MAPS),
}
manifest_file=TABLES/"scene12_processing_manifest.json"
manifest_file.write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
lines=[
    "# Notebook 12 — Per-scene Sentinel-1 and CHIRPS comparison",
    "",
    "**Analysis:** Scene-level radar VV low-backscatter screening, May–September 2022–2025.",
    "**Status:** Executed exploratory extraction; classification and hydrograph not validated.",
    f"**Scenes:** {len(scenes)}; support-eligible: {int(scenes.eligible.sum())}.",
    f"**EE polygon area:** {REFERENCE_KM2:.3f} km².",
    "",
    "## June per-acquisition VV screening range, by pass",
    "",
    "| Year | Pass | Scenes | Minimum −17 dB (km²) | Maximum −17 dB (km²) |",
    "|---|---|---:|---:|---:|",
]
for _, row in june_summary.iterrows():
    lines.append(f"| {int(row['year'])} | {row['pass']} | "
                 f"{int(row['n_scenes'])} | {row['minimum_m17_km2']:.3f} | {row['maximum_m17_km2']:.3f} |")
lines.extend([
    "",
    "## Interpretation",
    "",
    "These ranges are low-VV **scene screening** outputs, not flood area, depth, "
    "hazard or emissions. Scene support and viewing geometry differ. Acquisitions "
    "sample discrete dates and do not establish exact flood onset or duration.",
    "",
    "**Outstanding:** CHIRPS daily accumulation convention, independent SAR water/flooded-vegetation "
    "validation, river stage and upstream inflow, boundary-sensitive saturation, atmospheric methane observability.",
])
report_file=report_dir/"NOTEBOOK12_SCENE_LEVEL_FINDINGS.md"
report_file.write_text("\n".join(lines)+"\n",encoding="utf-8")
for p in (scene_csv,manifest_file,report_file,figure_file):
    if not p.exists():
        raise RuntimeError(f"Expected output absent: {p}")
    print("Saved:",p)
print("NOTE: These new results are local; not automatically uploaded or independently validated.")
display(june_summary)


## After running

Review individual scene support, the CHIRPS date-label sensitivity, orbit effects, and VV/VH distributions. If the observations look stable, pursue independently interpreted imagery and river-stage/upstream data. Do **not** revise the repository's 2022 or methane findings solely because Notebook 12 produces outputs. Publish the actual CSVs/figures/report only after local execution and QA.